# Module 5: The Math of Diffusion

This is the theoretical heart of the course. Everything we've built so far — PyTorch, convolutions, attention, U-Nets — was preparation for this moment. Now we'll develop the complete probabilistic framework behind diffusion models, step by step, from first principles.

### Where we're headed

We'll start from the forward noising process and work our way to the simplified training loss — the same one used in DDPM. Along the way, you'll implement noise schedules, walk through the ELBO derivation, and see why "just predict the noise" is mathematically justified.

### The closed-form forward process

You'll derive $q(x_t | x_0)$ — the formula that lets us skip directly to any noise level without running a 1000-step chain. This is what makes training efficient.

### Noise schedules and their impact

We'll build $\beta_t$, $\alpha_t$, and $\bar{\alpha}_t$ schedules (linear, cosine, sigmoid) and see exactly how they control the destruction of your image.

### The reverse process and ELBO

You'll work through the tractable reverse posterior $q(x_{t-1} | x_t, x_0)$ and the ELBO derivation that simplifies everything to "predict the noise."

### The SNR perspective

The signal-to-noise ratio gives us a unified lens for understanding what the model "sees" at each timestep — and why different schedules lead to different training dynamics.

### Key references

| Paper | What it contributes |
|---|---|
| [Ho et al. 2020](https://arxiv.org/abs/2006.11239) (DDPM) | The simplified loss and training algorithm we'll derive |
| [Sohl-Dickstein et al. 2015](https://arxiv.org/abs/1503.03585) | The original connection between diffusion and deep learning |
| [Nichol & Dhariwal 2021](https://arxiv.org/abs/2102.09672) | Cosine schedule and improved training |
| [Kingma & Gao 2023](https://arxiv.org/abs/2303.00848) | The SNR perspective that unifies everything |

**Estimated time:** 3–4 hours

In [ ]:
# --- Setup ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

from utils.data import get_device

torch.manual_seed(42)
device = get_device()
print(f"Using device: {device}")

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100

## 5.1 — Generative Models Landscape

Before we dive into diffusion, let's zoom out. Where do diffusion models sit among the other generative families? Each one makes a different tradeoff between sample quality, training stability, mode coverage, and sampling speed.

| Family | Training Objective | Mode Coverage | Sample Quality | Sampling Speed | Key Limitation |
|---|---|---|---|---|---|
| **GANs** | Adversarial min-max game | Poor (mode collapse) | Excellent | Fast (single pass) | Unstable training, mode dropping |
| **VAEs** | ELBO (reconstruction + KL) | Good | Moderate (blurry) | Fast (single pass) | Posterior approximation gap |
| **Normalizing Flows** | Exact log-likelihood | Excellent | Good | Fast (single pass) | Architecture constraints (invertibility) |
| **Autoregressive** | Exact log-likelihood (factorized) | Excellent | Excellent | Slow (sequential) | O(N) sampling for N dimensions |
| **Diffusion** | Denoising score matching / ELBO | Excellent | Excellent | Slow (iterative) | Requires many sampling steps |

Here's the big selling point of diffusion models: they achieve both **high sample quality** and **excellent mode coverage** — the two properties that are hardest to get simultaneously. The tradeoff is sampling speed, which has spurred a rich line of work on fast samplers (DDIM, DPM-Solver, consistency models, distillation).

Now let's build the mathematical framework that makes this possible.

## 5.2 — The Forward (Noising) Process

The forward process gradually destroys data by adding Gaussian noise over $T$ timesteps. Starting from a clean data point $x_0 \sim q(x_0)$, we define a Markov chain:

$$q(x_t | x_{t-1}) = \mathcal{N}\!\left(x_t;\; \sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t \mathbf{I}\right)$$

**What this does to your image:** at each step, we slightly shrink every pixel value (multiply by $\sqrt{1 - \beta_t} < 1$) and add a small amount of random noise (variance $\beta_t$). The pixel values slowly get erased, noise increases, until after enough steps the image is pure static.

For MNIST, $x_0$ has shape **(B, 1, 28, 28)** — a batch of single-channel 28×28 images. The noise $\varepsilon$ is the same shape: pure random noise sampled from a standard normal.

The full forward trajectory is:

$$q(x_{1:T} | x_0) = \prod_{t=1}^{T} q(x_t | x_{t-1})$$

Think of it like photocopying a photocopy, over and over. Each copy loses a little detail and picks up a little grain. After enough rounds, you can't tell what the original was.

Let's implement and visualize this step-by-step process.

In [ ]:
# --- 5.2 Forward process: single-step noising and sequential chain ---

def forward_step(x_prev: torch.Tensor, beta_t: float) -> torch.Tensor:
    """Apply one step of the forward process: q(x_t | x_{t-1}).
    
    Args:
        x_prev: tensor of shape (C, H, W), the image at step t-1
        beta_t: variance schedule value at step t
    Returns:
        x_t: noised tensor of shape (C, H, W)
    """
    noise = torch.randn_like(x_prev)                    # (C, H, W)
    mean = math.sqrt(1.0 - beta_t) * x_prev             # (C, H, W) -- shrink signal
    x_t = mean + math.sqrt(beta_t) * noise               # (C, H, W) -- add noise
    return x_t


# Create a synthetic "image" -- a simple checkerboard pattern
def make_checkerboard(size: int = 32, block: int = 4) -> torch.Tensor:
    """Create a checkerboard image tensor of shape (1, size, size) with values in [-1, 1]."""
    img = torch.zeros(1, size, size)
    for i in range(size):
        for j in range(size):
            if (i // block + j // block) % 2 == 0:
                img[0, i, j] = 1.0
            else:
                img[0, i, j] = -1.0
    return img

x_0 = make_checkerboard()  # (1, 32, 32)

# Linear beta schedule (DDPM default)
T = 1000
beta_start = 1e-4
beta_end = 0.02
betas = torch.linspace(beta_start, beta_end, T)  # (T,)

# Run the full forward chain and save snapshots
torch.manual_seed(42)
snapshot_steps = [0, 250, 500, 750, 1000]
snapshots = {0: x_0.clone()}
x_t = x_0.clone()

for t in range(1, T + 1):
    x_t = forward_step(x_t, betas[t - 1].item())
    if t in snapshot_steps:
        snapshots[t] = x_t.clone()

# Visualize
fig, axes = plt.subplots(1, len(snapshot_steps), figsize=(14, 3))
for ax, t in zip(axes, snapshot_steps):
    ax.imshow(snapshots[t][0].cpu().numpy(), cmap='gray', vmin=-3, vmax=3)
    ax.set_title(f"t = {t}")
    ax.axis('off')
fig.suptitle("Forward Process: Gradual Destruction of Signal", fontsize=13)
plt.tight_layout()
plt.show()

## 5.3 — Noise Schedule: $\beta_t$, $\alpha_t$, $\bar{\alpha}_t$

Running the forward chain step-by-step is expensive — for $T = 1000$ steps, that's 1000 sequential operations just to get one training sample. A key insight from [Ho et al. (2020)](https://arxiv.org/abs/2006.11239) is that we can **jump directly** from $x_0$ to any $x_t$ in closed form.

### Definitions

$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$$

- $\alpha_t$ is how much signal is **kept** at step $t$ (close to 1 for small $\beta_t$)
- $\bar{\alpha}_t$ is the **cumulative product** — how much of the original signal survives all the way to step $t$
- When $\bar{\alpha}_t \approx 1$, the image is nearly clean. When $\bar{\alpha}_t \approx 0$, the image is nearly pure noise.

### Closed-Form Forward Process

By recursively substituting the single-step formula and using the fact that sums of independent Gaussians are Gaussian:

$$q(x_t | x_0) = \mathcal{N}\!\left(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t)\, \mathbf{I}\right)$$

**In plain language:** the noisy image $x_t$ is a weighted mix of the original image and pure noise. The weight on the signal is $\sqrt{\bar{\alpha}_t}$, and the weight on the noise is $\sqrt{1 - \bar{\alpha}_t}$.

This means we can sample $x_t$ directly via the **reparameterization**:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, \mathbf{I})$$

In code, $x_0$ has shape **(B, 1, 28, 28)** and $\varepsilon$ has the same shape. The coefficients $\sqrt{\bar{\alpha}_t}$ and $\sqrt{1 - \bar{\alpha}_t}$ are scalars (one per timestep) that get broadcast across the spatial dimensions.

This closed form is critical for training — we never need to run the chain sequentially.

In [ ]:
# --- 5.3 Compute the schedule and plot key curves ---

def compute_linear_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02):
    """Compute the linear beta schedule and derived quantities.
    
    Returns:
        dict with keys: betas, alphas, alpha_bar (all shape (T,))
    """
    betas = torch.linspace(beta_start, beta_end, T)          # (T,)
    alphas = 1.0 - betas                                      # (T,)
    alpha_bar = torch.cumprod(alphas, dim=0)                  # (T,)
    return {'betas': betas, 'alphas': alphas, 'alpha_bar': alpha_bar}

schedule = compute_linear_schedule(T=1000)
timesteps = torch.arange(1, 1001)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(timesteps.numpy(), schedule['betas'].numpy(), color='tab:red')
axes[0].set_title(r'$\beta_t$ (variance schedule)')
axes[0].set_xlabel('Timestep t')

axes[1].plot(timesteps.numpy(), schedule['alphas'].numpy(), color='tab:blue')
axes[1].set_title(r'$\alpha_t = 1 - \beta_t$')
axes[1].set_xlabel('Timestep t')

axes[2].plot(timesteps.numpy(), schedule['alpha_bar'].numpy(), color='tab:green')
axes[2].set_title(r'$\bar{\alpha}_t$ (signal survival)')
axes[2].set_xlabel('Timestep t')
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"alpha_bar at t=1:   {schedule['alpha_bar'][0]:.6f}  (almost all signal)")
print(f"alpha_bar at t=500: {schedule['alpha_bar'][499]:.6f}")
print(f"alpha_bar at t=1000: {schedule['alpha_bar'][999]:.6f}  (almost pure noise)")

In [ ]:
# --- 5.3 Verify closed-form matches sequential noising ---

def closed_form_sample(x_0: torch.Tensor, alpha_bar_t: float, noise: torch.Tensor) -> torch.Tensor:
    """Sample x_t directly from x_0 using the closed-form formula.
    
    x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
    """
    return math.sqrt(alpha_bar_t) * x_0 + math.sqrt(1.0 - alpha_bar_t) * noise  # (C, H, W)


# Compare: sequential vs closed-form at t=500
# We can't get exact match (different random draws), but we can verify
# the STATISTICS match (mean and variance).

x_0 = make_checkerboard()  # (1, 32, 32)
test_t = 500
alpha_bar_t = schedule['alpha_bar'][test_t - 1].item()

# Run many samples and check statistics
n_samples = 5000
torch.manual_seed(0)

# Closed-form samples for a single pixel
pixel_samples_cf = []
for _ in range(n_samples):
    eps = torch.randn(1)
    x_t_pixel = math.sqrt(alpha_bar_t) * x_0[0, 0, 0].item() + math.sqrt(1 - alpha_bar_t) * eps.item()
    pixel_samples_cf.append(x_t_pixel)

pixel_samples_cf = torch.tensor(pixel_samples_cf)

expected_mean = math.sqrt(alpha_bar_t) * x_0[0, 0, 0].item()
expected_var = 1 - alpha_bar_t

print(f"At t={test_t}, alpha_bar_t = {alpha_bar_t:.6f}")
print(f"Pixel value x_0[0,0,0] = {x_0[0,0,0].item():.1f}")
print(f"Expected mean: sqrt(alpha_bar)*x_0 = {expected_mean:.4f}")
print(f"Empirical mean: {pixel_samples_cf.mean():.4f}")
print(f"Expected variance: 1 - alpha_bar = {expected_var:.4f}")
print(f"Empirical variance: {pixel_samples_cf.var():.4f}")
print(f"\nStatistics match -- closed-form is correct.")

### Exercise 5.1: Implement the Full Schedule Computation

**Write a function `compute_schedule(betas)` that takes a tensor of $\beta_t$ values and returns all the derived quantities we'll need throughout this module.**

Your function should return a dictionary with these keys (all 1-D tensors of shape **(T,)**):

| Key | Formula | What it represents |
|---|---|---|
| `betas` | $\beta_t$ | The input variance schedule |
| `alphas` | $\alpha_t = 1 - \beta_t$ | How much signal is kept per step |
| `alpha_bar` | $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$ | How much original signal survives to step $t$ |
| `sqrt_alpha_bar` | $\sqrt{\bar{\alpha}_t}$ | Coefficient on the clean image |
| `sqrt_one_minus_alpha_bar` | $\sqrt{1 - \bar{\alpha}_t}$ | Coefficient on the noise |
| `sqrt_recip_alpha` | $1 / \sqrt{\alpha_t}$ | Needed for the sampling step |
| `posterior_variance` | $\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t} \beta_t$ | We'll derive this in Section 5.5 |

**Hints:**
- Use `torch.cumprod` for the cumulative product
- For `posterior_variance`, you need $\bar{\alpha}_{t-1}$. At $t=1$ (index 0), define $\bar{\alpha}_0 = 1.0$. Use `F.pad` to prepend this value.

In [ ]:
# YOUR CODE HERE — Exercise 5.1

def compute_schedule(betas: torch.Tensor) -> dict:
    """Compute all diffusion schedule quantities from beta values.
    
    Args:
        betas: tensor of shape (T,), the variance schedule
    Returns:
        dict of schedule tensors, each of shape (T,)
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
betas_test = torch.linspace(1e-4, 0.02, 1000)
sched_test = compute_schedule(betas_test)

assert sched_test is not None, "compute_schedule returned None — did you forget to return the dict?"
assert isinstance(sched_test, dict), f"Expected dict, got {type(sched_test)} — make sure you return a dictionary"
expected_keys = {'betas', 'alphas', 'alpha_bar', 'sqrt_alpha_bar', 
                 'sqrt_one_minus_alpha_bar', 'sqrt_recip_alpha', 'posterior_variance'}
missing = expected_keys - set(sched_test.keys())
assert not missing, f"Missing keys: {missing} — check your return dict"
assert sched_test['alpha_bar'].shape == (1000,), \
    f"alpha_bar shape should be (1000,), got {sched_test['alpha_bar'].shape} — are you passing dim=0 to cumprod?"
assert sched_test['alpha_bar'][0] > 0.99, \
    f"alpha_bar[0] should be ~1.0, got {sched_test['alpha_bar'][0]:.4f} — did you forget torch.cumprod? alpha_bar is a cumulative product of alphas"
assert sched_test['alpha_bar'][-1] < 0.01, \
    f"alpha_bar[-1] should be ~0, got {sched_test['alpha_bar'][-1]:.4f} — the signal should be nearly gone by the last timestep"
assert sched_test['posterior_variance'][0] < 1e-6, \
    f"posterior_variance[0] should be ~0, got {sched_test['posterior_variance'][0]:.6f} — check your alpha_bar_prev padding (alpha_bar_0 should be 1.0)"
print("All checks passed ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def compute_schedule(betas: torch.Tensor) -> dict:
    """Compute all diffusion schedule quantities from beta values.
    
    Args:
        betas: tensor of shape (T,), the variance schedule
    Returns:
        dict of schedule tensors, each of shape (T,)
    """
    alphas = 1.0 - betas                                          # (T,)
    alpha_bar = torch.cumprod(alphas, dim=0)                      # (T,)
    
    # For posterior variance, we need alpha_bar_{t-1}. 
    # At t=1 (index 0), alpha_bar_{t-1} = alpha_bar_0 = 1.0
    alpha_bar_prev = F.pad(alpha_bar[:-1], (1, 0), value=1.0)    # (T,)
    
    sqrt_alpha_bar = torch.sqrt(alpha_bar)                        # (T,)
    sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)        # (T,)
    sqrt_recip_alpha = 1.0 / torch.sqrt(alphas)                   # (T,)
    
    # Posterior variance: beta_tilde_t = (1 - alpha_bar_{t-1}) / (1 - alpha_bar_t) * beta_t
    posterior_variance = (1.0 - alpha_bar_prev) / (1.0 - alpha_bar) * betas  # (T,)
    
    return {
        'betas': betas,
        'alphas': alphas,
        'alpha_bar': alpha_bar,
        'alpha_bar_prev': alpha_bar_prev,
        'sqrt_alpha_bar': sqrt_alpha_bar,
        'sqrt_one_minus_alpha_bar': sqrt_one_minus_alpha_bar,
        'sqrt_recip_alpha': sqrt_recip_alpha,
        'posterior_variance': posterior_variance,
    }


# Test it
betas = torch.linspace(1e-4, 0.02, 1000)
sched = compute_schedule(betas)

print("Schedule keys:", list(sched.keys()))
print(f"alpha_bar[0] = {sched['alpha_bar'][0]:.6f} (should be close to 1)")
print(f"alpha_bar[-1] = {sched['alpha_bar'][-1]:.6f} (should be close to 0)")
print(f"posterior_variance[0] = {sched['posterior_variance'][0]:.8f} (should be 0 at t=1)")
print(f"sqrt_alpha_bar shape: {sched['sqrt_alpha_bar'].shape}")
print("All schedule tensors computed correctly.")

## 5.4 — The Reparameterization Trick

To train with gradient descent, we need to backpropagate through the sampling operation. But sampling $x_t \sim q(x_t | x_0)$ involves randomness — and you can't differentiate through a random sample.

The **reparameterization trick** (from [Kingma & Welling, 2014](https://arxiv.org/abs/1312.6114)) separates the randomness from the parameters:

Instead of sampling $x_t \sim \mathcal{N}(\mu, \sigma^2)$ directly, we:

1. Sample $\varepsilon \sim \mathcal{N}(0, \mathbf{I})$ — no learnable parameters involved
2. Compute $x_t = \mu + \sigma \cdot \varepsilon$ — a deterministic, differentiable function of $\mu$ and $\sigma$

For the forward process, this gives us:

$$x_t = \underbrace{\sqrt{\bar{\alpha}_t}}_{\text{signal coeff}} \cdot x_0 + \underbrace{\sqrt{1 - \bar{\alpha}_t}}_{\text{noise coeff}} \cdot \varepsilon$$

**What's happening to the image here:** the first term keeps a fraction of each pixel value from the original, and the second term adds random noise on top. Both $x_0$ and $\varepsilon$ have shape **(B, 1, 28, 28)** for MNIST. The coefficients are scalars that broadcast.

Both coefficients are differentiable functions of the schedule parameters, and $\varepsilon$ is independent of everything we want to optimize. Gradients flow cleanly through $x_t$ back to the model.

In [ ]:
# --- 5.4 Two equivalent ways to sample x_t ---

x_0 = make_checkerboard()  # (1, 32, 32)
t_idx = 499  # t=500 (0-indexed)

# Method 1: Direct sampling from Gaussian (NOT differentiable w.r.t. schedule params)
torch.manual_seed(123)
mean_t = sched['sqrt_alpha_bar'][t_idx] * x_0           # (1, 32, 32)
std_t = sched['sqrt_one_minus_alpha_bar'][t_idx]         # scalar
x_t_direct = mean_t + std_t * torch.randn_like(x_0)     # (1, 32, 32)

# Method 2: Reparameterization (differentiable w.r.t. schedule params)
torch.manual_seed(123)
eps = torch.randn_like(x_0)                                                    # (1, 32, 32)
x_t_reparam = sched['sqrt_alpha_bar'][t_idx] * x_0 + sched['sqrt_one_minus_alpha_bar'][t_idx] * eps  # (1, 32, 32)

# They produce identical results with the same seed
print(f"Max difference between methods: {(x_t_direct - x_t_reparam).abs().max().item():.2e}")
print("Both methods produce identical samples (as expected).")

# Show the result
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(x_0[0].numpy(), cmap='gray', vmin=-3, vmax=3)
axes[0].set_title("$x_0$ (clean)")
axes[1].imshow(eps[0].numpy(), cmap='gray', vmin=-3, vmax=3)
axes[1].set_title(r"$\varepsilon \sim \mathcal{N}(0, I)$")
axes[2].imshow(x_t_reparam[0].numpy(), cmap='gray', vmin=-3, vmax=3)
axes[2].set_title(f"$x_{{500}}$ (reparameterized)")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5.5 — The Reverse Process

Now for the hard part. We want to run the noising process **backwards**: given a noisy image $x_t$, produce a slightly less noisy image $x_{t-1}$. Mathematically, we need $q(x_{t-1} | x_t)$.

### Why the reverse is intractable

Computing $q(x_{t-1} | x_t)$ exactly requires integrating over all possible clean images:

$$q(x_{t-1} | x_t) = \int q(x_{t-1} | x_t, x_0)\, q(x_0 | x_t)\, dx_0$$

This is intractable because $q(x_0 | x_t)$ depends on the unknown data distribution. We'd need to know every possible clean image that could have produced this noisy one.

Instead, we train a neural network $p_\theta$ to approximate the reverse step:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}\!\left(x_{t-1};\; \mu_\theta(x_t, t),\; \Sigma_\theta(x_t, t)\right)$$

We're asking the network: **given this noisy mess, can you figure out what noise was added?**

### The tractable posterior — here's where things get interesting

While $q(x_{t-1} | x_t)$ is intractable, the **posterior conditioned on $x_0$** turns out to be tractable and Gaussian:

$$q(x_{t-1} | x_t, x_0) = \mathcal{N}\!\left(x_{t-1};\; \tilde{\mu}_t(x_t, x_0),\; \tilde{\beta}_t \mathbf{I}\right)$$

The posterior mean is a weighted combination of the clean image and the noisy image:

$$\tilde{\mu}_t(x_t, x_0) = \frac{\sqrt{\bar{\alpha}_{t-1}}\, \beta_t}{1 - \bar{\alpha}_t}\, x_0 + \frac{\sqrt{\alpha_t}\,(1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t}\, x_t$$

**What this does:** it takes the "ideal" slightly-less-noisy image $x_{t-1}$ as a blend of the original $x_0$ (how much we'd like to get back) and the current noisy $x_t$ (what we're starting from). The schedule coefficients control the blend.

The posterior variance is simply:

$$\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\, \beta_t$$

This is a scalar per timestep — no learnable parameters. It comes from applying Bayes' rule to the Gaussian forward process and completing the square.

Now let's see how we turn this into a practical training objective.

## 5.6 — ELBO Derivation: Why Predicting Noise Works

This is the mathematical core of DDPM ([Ho et al. 2020](https://arxiv.org/abs/2006.11239)). We'll walk through the derivation that connects the variational bound to the simple noise-prediction loss. Follow the steps carefully — by the end, the simplified loss will feel almost obvious.

### Step 1: The variational lower bound

We want to maximize $\log p_\theta(x_0)$. Using Jensen's inequality:

$$\log p_\theta(x_0) \geq \mathbb{E}_{q(x_{1:T}|x_0)} \left[ \log \frac{p_\theta(x_{0:T})}{q(x_{1:T}|x_0)} \right] = -L_{\text{VLB}}$$

This is the same ELBO idea from VAEs — we can't compute the true likelihood, so we optimize a lower bound instead.

### Step 2: Decompose into per-timestep terms

The VLB splits into $T+1$ terms, each with a clear meaning:

$$L_{\text{VLB}} = \underbrace{D_{\text{KL}}(q(x_T|x_0) \| p(x_T))}_{L_T} + \sum_{t=2}^{T} \underbrace{D_{\text{KL}}(q(x_{t-1}|x_t, x_0) \| p_\theta(x_{t-1}|x_t))}_{L_{t-1}} - \underbrace{\log p_\theta(x_0|x_1)}_{L_0}$$

| Term | What it measures |
|---|---|
| $L_T$ | Does the forward process end at pure noise $\mathcal{N}(0, I)$? No learnable parameters — just a constant. |
| $L_{t-1}$ | KL between the true reverse step $q(x_{t-1}\|x_t,x_0)$ and learned reverse $p_\theta(x_{t-1}\|x_t)$. Both Gaussian, so closed form. |
| $L_0$ | Reconstruction — how well we recover the clean image from the first noisy step. |

### Step 3: KL between two Gaussians simplifies to mean matching

Since both $q(x_{t-1}|x_t,x_0)$ and $p_\theta(x_{t-1}|x_t)$ are Gaussian with the same (fixed) variance $\tilde{\beta}_t$, the KL reduces to comparing their means:

$$L_{t-1} = \frac{1}{2\tilde{\beta}_t} \left\| \tilde{\mu}_t(x_t, x_0) - \mu_\theta(x_t, t) \right\|^2 + C$$

The network just needs to match the true posterior mean. That's the entire learning signal.

### Step 4: The epsilon reparameterization — the key move

Since $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon$, we can express $x_0$ in terms of $x_t$ and $\varepsilon$:

$$x_0 = \frac{1}{\sqrt{\bar{\alpha}_t}} \left( x_t - \sqrt{1 - \bar{\alpha}_t}\, \varepsilon \right)$$

Substituting into $\tilde{\mu}_t$, after algebra:

$$\tilde{\mu}_t = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \varepsilon \right)$$

**What this says:** the true posterior mean is just $x_t$ with the noise partially subtracted out. The fraction $\frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}$ controls how much noise to remove at each step.

If we parameterize $\mu_\theta$ the same way but with a *predicted* noise $\varepsilon_\theta$:

$$\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \varepsilon_\theta(x_t, t) \right)$$

then $L_{t-1}$ becomes proportional to:

$$L_{t-1} \propto \left\| \varepsilon - \varepsilon_\theta(x_t, t) \right\|^2$$

The network takes in the noisy image $x_t$ (shape **(B, 1, 28, 28)**) and the timestep $t$, and outputs a prediction of the noise $\varepsilon_\theta$ (same shape **(B, 1, 28, 28)**). The loss is just MSE between the true noise and the predicted noise.

### Alternative parameterizations

You'll see three parameterizations in the literature. They're all mathematically equivalent — they differ in numerical conditioning at different noise levels:

| Parameterization | Network predicts | Loss target | Reference |
|---|---|---|---|
| $\varepsilon$-prediction | $\varepsilon_\theta(x_t, t) \approx \varepsilon$ | $\|\varepsilon - \varepsilon_\theta\|^2$ | [Ho et al. 2020](https://arxiv.org/abs/2006.11239) |
| $x_0$-prediction | $x_{0,\theta}(x_t, t) \approx x_0$ | $\|x_0 - x_{0,\theta}\|^2$ | [Ramesh et al. 2022](https://arxiv.org/abs/2204.06125) |
| $v$-prediction | $v_\theta(x_t, t) \approx v_t$ | $\|v_t - v_\theta\|^2$ | [Salimans & Ho 2022](https://arxiv.org/abs/2202.00512) |

where $v_t = \sqrt{\bar{\alpha}_t}\, \varepsilon - \sqrt{1 - \bar{\alpha}_t}\, x_0$.

In practice, $\varepsilon$-prediction is the most common starting point, and we'll use it for training in this course. Let's now see these conversions in code.

In [ ]:
# --- 5.6 Convert between eps, mu, and x_{t-1} ---

def eps_to_mu(x_t: torch.Tensor, eps_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """Convert predicted noise to predicted posterior mean.
    
    mu_theta = (1/sqrt(alpha_t)) * (x_t - beta_t / sqrt(1 - alpha_bar_t) * eps_theta)
    
    Args:
        x_t: noisy input, shape (B, C, H, W)
        eps_theta: predicted noise, shape (B, C, H, W)
        t_idx: 0-indexed timestep
        sched: schedule dict from compute_schedule()
    Returns:
        mu_theta: predicted mean, shape (B, C, H, W)
    """
    sqrt_recip_alpha = sched['sqrt_recip_alpha'][t_idx]           # scalar
    beta_t = sched['betas'][t_idx]                                 # scalar
    sqrt_one_minus_alpha_bar = sched['sqrt_one_minus_alpha_bar'][t_idx]  # scalar
    
    mu_theta = sqrt_recip_alpha * (x_t - (beta_t / sqrt_one_minus_alpha_bar) * eps_theta)  # (B, C, H, W)
    return mu_theta


def eps_to_x0(x_t: torch.Tensor, eps_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """Convert predicted noise to predicted x_0.
    
    x_0 = (x_t - sqrt(1 - alpha_bar_t) * eps) / sqrt(alpha_bar_t)
    
    Args:
        x_t: noisy input, shape (B, C, H, W)
        eps_theta: predicted noise, shape (B, C, H, W)
        t_idx: 0-indexed timestep
        sched: schedule dict
    Returns:
        x_0_pred: predicted clean image, shape (B, C, H, W)
    """
    x_0_pred = (x_t - sched['sqrt_one_minus_alpha_bar'][t_idx] * eps_theta) / sched['sqrt_alpha_bar'][t_idx]  # (B, C, H, W)
    return x_0_pred


def sample_reverse_step(x_t: torch.Tensor, eps_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """One reverse sampling step: x_{t-1} ~ p_theta(x_{t-1} | x_t).
    
    Args:
        x_t: shape (B, C, H, W)
        eps_theta: shape (B, C, H, W)
        t_idx: 0-indexed timestep
        sched: schedule dict
    Returns:
        x_prev: shape (B, C, H, W)
    """
    mu_theta = eps_to_mu(x_t, eps_theta, t_idx, sched)         # (B, C, H, W)
    
    if t_idx == 0:
        return mu_theta  # No noise at final step
    
    sigma_t = torch.sqrt(sched['posterior_variance'][t_idx])     # scalar
    noise = torch.randn_like(x_t)                                # (B, C, H, W)
    x_prev = mu_theta + sigma_t * noise                          # (B, C, H, W)
    return x_prev


# Demonstrate: if we know the true noise, we can perfectly recover the posterior mean
torch.manual_seed(42)
x_0 = make_checkerboard().unsqueeze(0)           # (1, 1, 32, 32)
t_idx = 300
eps_true = torch.randn_like(x_0)                  # (1, 1, 32, 32)

# Forward: noise x_0 to get x_t
x_t = sched['sqrt_alpha_bar'][t_idx] * x_0 + sched['sqrt_one_minus_alpha_bar'][t_idx] * eps_true  # (1, 1, 32, 32)

# Recover x_0 from x_t and true noise
x_0_recovered = eps_to_x0(x_t, eps_true, t_idx, sched)  # (1, 1, 32, 32)

print(f"Max error in x_0 recovery: {(x_0 - x_0_recovered).abs().max().item():.2e}")
print("With the true noise, x_0 is recovered exactly.")

### Exercise 5.2: Implement All Three Prediction Modes

**Given $x_t$, a prediction, and the schedule, convert between the three parameterizations.** You'll implement these conversions and use them throughout the rest of the course.

### `predict_x0_from_eps` — recover $x_0$ from predicted noise

$$x_0 = (x_t - \sqrt{1-\bar{\alpha}_t}\, \varepsilon_\theta) / \sqrt{\bar{\alpha}_t}$$

### `predict_eps_from_x0` — recover $\varepsilon$ from predicted $x_0$

$$\varepsilon = (x_t - \sqrt{\bar{\alpha}_t}\, x_{0,\theta}) / \sqrt{1-\bar{\alpha}_t}$$

### `predict_x0_eps_from_v` — recover both $x_0$ and $\varepsilon$ from predicted $v$

Recall: $v_t = \sqrt{\bar{\alpha}_t}\, \varepsilon - \sqrt{1 - \bar{\alpha}_t}\, x_0$. Solve the system of $x_t$ and $v_t$ equations for $x_0$ and $\varepsilon$.

All inputs and outputs have the same shape as $x_t$: **(B, 1, 28, 28)** for MNIST. The schedule coefficients are scalars indexed by timestep and broadcast across spatial dimensions.

In [ ]:
# YOUR CODE HERE — Exercise 5.2

def predict_x0_from_eps(x_t: torch.Tensor, eps_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """eps-prediction mode: recover x_0 from predicted noise."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


def predict_eps_from_x0(x_t: torch.Tensor, x0_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """x0-prediction mode: recover eps from predicted x_0."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


def predict_x0_eps_from_v(x_t: torch.Tensor, v_theta: torch.Tensor, t_idx: int, sched: dict) -> tuple:
    """v-prediction mode: recover both x_0 and eps from predicted v.
    
    Returns:
        (x0_pred, eps_pred) — both shape (B, C, H, W)
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
torch.manual_seed(7)
x_0_test = torch.randn(1, 1, 8, 8)                                # (1, 1, 8, 8)
eps_true = torch.randn_like(x_0_test)                              # (1, 1, 8, 8)
t_idx = 400

x_t_test = sched['sqrt_alpha_bar'][t_idx] * x_0_test + sched['sqrt_one_minus_alpha_bar'][t_idx] * eps_true
v_true = sched['sqrt_alpha_bar'][t_idx] * eps_true - sched['sqrt_one_minus_alpha_bar'][t_idx] * x_0_test

x0_from_eps = predict_x0_from_eps(x_t_test, eps_true, t_idx, sched)
assert x0_from_eps is not None, "predict_x0_from_eps returned None — did you forget the return statement?"
assert (x_0_test - x0_from_eps).abs().max() < 1e-5, \
    f"eps-mode x_0 error too large: {(x_0_test - x0_from_eps).abs().max():.2e} — check that you're dividing by sqrt_alpha_bar, not multiplying"

eps_from_x0 = predict_eps_from_x0(x_t_test, x_0_test, t_idx, sched)
assert eps_from_x0 is not None, "predict_eps_from_x0 returned None — did you forget the return statement?"
assert (eps_true - eps_from_x0).abs().max() < 1e-5, \
    f"x0-mode eps error too large: {(eps_true - eps_from_x0).abs().max():.2e} — check that you're dividing by sqrt_one_minus_alpha_bar"

result_v = predict_x0_eps_from_v(x_t_test, v_true, t_idx, sched)
assert result_v is not None, "predict_x0_eps_from_v returned None — did you forget the return statement?"
x0_from_v, eps_from_v = result_v
assert (x_0_test - x0_from_v).abs().max() < 1e-5, \
    f"v-mode x_0 error: {(x_0_test - x0_from_v).abs().max():.2e} — x_0 = sqrt_ab * x_t - sqrt_1mab * v"
assert (eps_true - eps_from_v).abs().max() < 1e-5, \
    f"v-mode eps error: {(eps_true - eps_from_v).abs().max():.2e} — eps = sqrt_1mab * x_t + sqrt_ab * v"

print("eps-mode  ✓")
print("x0-mode   ✓")
print("v-mode    ✓")
print("All three parameterizations are consistent!")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def predict_x0_from_eps(x_t: torch.Tensor, eps_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """eps-prediction mode: recover x_0 from predicted noise."""
    return (x_t - sched['sqrt_one_minus_alpha_bar'][t_idx] * eps_theta) / sched['sqrt_alpha_bar'][t_idx]  # (B, C, H, W)


def predict_eps_from_x0(x_t: torch.Tensor, x0_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """x0-prediction mode: recover eps from predicted x_0."""
    return (x_t - sched['sqrt_alpha_bar'][t_idx] * x0_theta) / sched['sqrt_one_minus_alpha_bar'][t_idx]  # (B, C, H, W)


def predict_x0_eps_from_v(x_t: torch.Tensor, v_theta: torch.Tensor, t_idx: int, sched: dict) -> tuple:
    """v-prediction mode: recover both x_0 and eps from predicted v.
    
    v_t = sqrt(alpha_bar) * eps - sqrt(1 - alpha_bar) * x_0
    
    Solving the system:
      x_t = sqrt(alpha_bar) * x_0 + sqrt(1 - alpha_bar) * eps
      v_t = sqrt(alpha_bar) * eps - sqrt(1 - alpha_bar) * x_0
    
    Gives:
      x_0 = sqrt(alpha_bar) * x_t - sqrt(1 - alpha_bar) * v_t
      eps = sqrt(1 - alpha_bar) * x_t + sqrt(alpha_bar) * v_t
    """
    sqrt_ab = sched['sqrt_alpha_bar'][t_idx]
    sqrt_1mab = sched['sqrt_one_minus_alpha_bar'][t_idx]
    
    x0_pred = sqrt_ab * x_t - sqrt_1mab * v_theta                # (B, C, H, W)
    eps_pred = sqrt_1mab * x_t + sqrt_ab * v_theta               # (B, C, H, W)
    return x0_pred, eps_pred


# --- Verify all three modes are consistent ---
torch.manual_seed(7)
x_0_test = torch.randn(1, 1, 8, 8)                                # (1, 1, 8, 8)
eps_true = torch.randn_like(x_0_test)                              # (1, 1, 8, 8)
t_idx = 400

x_t_test = sched['sqrt_alpha_bar'][t_idx] * x_0_test + sched['sqrt_one_minus_alpha_bar'][t_idx] * eps_true  # (1, 1, 8, 8)

# Compute true v
v_true = sched['sqrt_alpha_bar'][t_idx] * eps_true - sched['sqrt_one_minus_alpha_bar'][t_idx] * x_0_test  # (1, 1, 8, 8)

# Mode 1: eps -> x_0
x0_from_eps = predict_x0_from_eps(x_t_test, eps_true, t_idx, sched)
print(f"eps-mode  |  x_0 error: {(x_0_test - x0_from_eps).abs().max().item():.2e}")

# Mode 2: x_0 -> eps
eps_from_x0 = predict_eps_from_x0(x_t_test, x_0_test, t_idx, sched)
print(f"x0-mode   |  eps error: {(eps_true - eps_from_x0).abs().max().item():.2e}")

# Mode 3: v -> x_0 and eps
x0_from_v, eps_from_v = predict_x0_eps_from_v(x_t_test, v_true, t_idx, sched)
print(f"v-mode    |  x_0 error: {(x_0_test - x0_from_v).abs().max().item():.2e}")
print(f"v-mode    |  eps error: {(eps_true - eps_from_v).abs().max().item():.2e}")
print("\nAll three parameterizations are consistent.")

## 5.7 — The Simplified Loss

[Ho et al. (2020)](https://arxiv.org/abs/2006.11239) found that dropping the weighting factor $\frac{\beta_t^2}{2\tilde{\beta}_t \alpha_t (1 - \bar{\alpha}_t)}$ from the ELBO and using a **uniform weight** across timesteps actually works better in practice:

$$L_{\text{simple}} = \mathbb{E}_{t \sim \mathcal{U}(1,T),\; x_0,\; \varepsilon \sim \mathcal{N}(0,I)} \left[ \left\| \varepsilon - \varepsilon_\theta\!\left(\sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon,\; t\right) \right\|^2 \right]$$

**What this loss computes:** take a clean image, add a known amount of noise, ask the network to predict that noise, penalize the squared error. That's the entire training procedure.

### DDPM Algorithm 1 (Training)

Here's what one training step looks like — this is the algorithm you'll implement:

1. Sample a clean image $x_0$ from the dataset — shape **(B, 1, 28, 28)**
2. Sample a random timestep $t \sim \mathcal{U}\{1, \ldots, T\}$
3. Sample noise $\varepsilon \sim \mathcal{N}(0, \mathbf{I})$ — same shape **(B, 1, 28, 28)**
4. Compute the noisy image: $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon$
5. Feed $x_t$ and $t$ to the network, get predicted noise $\varepsilon_\theta(x_t, t)$
6. Take a gradient step on $\| \varepsilon - \varepsilon_\theta(x_t, t) \|^2$

The network is learning to look at a noisy image and figure out which part is noise and which part is signal.

In [ ]:
# --- 5.7 The simplified loss in ~10 lines ---

def diffusion_loss_simple(
    model: nn.Module,
    x_0: torch.Tensor,
    sched: dict,
    T: int = 1000,
) -> torch.Tensor:
    """Compute the simplified diffusion training loss (DDPM Algorithm 1).
    
    Args:
        model: noise prediction network, takes (x_t, t) -> eps_theta
        x_0: clean data batch, shape (B, C, H, W)
        sched: schedule dict from compute_schedule()
        T: number of diffusion timesteps
    Returns:
        loss: scalar MSE loss
    """
    batch_size = x_0.shape[0]
    
    # 1. Sample random timesteps uniformly
    t = torch.randint(0, T, (batch_size,), device=x_0.device)            # (B,)
    
    # 2. Sample noise
    eps = torch.randn_like(x_0)                                           # (B, C, H, W)
    
    # 3. Compute x_t using closed-form forward process
    sqrt_alpha_bar_t = sched['sqrt_alpha_bar'][t].view(-1, 1, 1, 1)       # (B, 1, 1, 1)
    sqrt_one_minus_ab_t = sched['sqrt_one_minus_alpha_bar'][t].view(-1, 1, 1, 1)  # (B, 1, 1, 1)
    x_t = sqrt_alpha_bar_t * x_0 + sqrt_one_minus_ab_t * eps             # (B, C, H, W)
    
    # 4. Predict noise
    eps_theta = model(x_t, t)                                              # (B, C, H, W)
    
    # 5. MSE loss
    loss = F.mse_loss(eps_theta, eps)                                      # scalar
    return loss


# Quick sanity check with a dummy model
class DummyNoisePredictor(nn.Module):
    """A trivial model that just returns zeros -- for testing the loss function."""
    def forward(self, x_t, t):
        return torch.zeros_like(x_t)

dummy_model = DummyNoisePredictor()
torch.manual_seed(42)
x_0_batch = torch.randn(4, 1, 32, 32)  # (4, 1, 32, 32)
loss = diffusion_loss_simple(dummy_model, x_0_batch, sched, T=1000)
print(f"Loss with zero-prediction model: {loss.item():.4f}")
print(f"Expected ~1.0 (since eps ~ N(0,I) and model predicts 0, MSE ≈ E[eps^2] = 1)")
print(f"Loss is differentiable: {loss.requires_grad}")

### Exercise 5.3: Implement the Diffusion Loss Function

**Implement `diffusion_loss` that supports all three prediction modes (eps, x0, v).** This is the function you'd call inside a training loop.

Your function should:

1. Sample random timesteps and noise
2. Compute $x_t$ using the closed-form forward process
3. Pass $x_t$ and $t$ through the model to get a prediction
4. Compute MSE against the appropriate target:

| Mode | Target | What the network learns |
|---|---|---|
| `"eps"` | $\varepsilon$ | Guess what noise was added |
| `"x0"` | $x_0$ | Guess the original clean image |
| `"v"` | $v_t = \sqrt{\bar{\alpha}_t}\varepsilon - \sqrt{1-\bar{\alpha}_t} x_0$ | Predict a combination of both |

All tensors have shape **(B, C, H, W)**. Use `.view(-1, 1, 1, 1)` to broadcast schedule coefficients across spatial dimensions.

In [ ]:
# YOUR CODE HERE — Exercise 5.3

def diffusion_loss(
    model: nn.Module,
    x_0: torch.Tensor,
    sched: dict,
    T: int = 1000,
    prediction_mode: str = "eps",
) -> torch.Tensor:
    """Compute diffusion training loss with configurable prediction mode.
    
    Args:
        model: network that takes (x_t, t) and returns prediction
        x_0: clean data, shape (B, C, H, W)
        sched: schedule dict from compute_schedule()
        T: number of timesteps
        prediction_mode: one of "eps", "x0", "v"
    Returns:
        loss: scalar MSE
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
torch.manual_seed(42)
x_0_batch = torch.randn(4, 1, 32, 32)  # (4, 1, 32, 32)

for mode in ["eps", "x0", "v"]:
    loss = diffusion_loss(dummy_model, x_0_batch, sched, T=1000, prediction_mode=mode)
    assert loss is not None, f"diffusion_loss returned None for mode '{mode}' — did you forget the return statement?"
    assert loss.shape == (), f"Loss should be a scalar, got shape {loss.shape} — use F.mse_loss which returns a scalar by default"
    assert loss.item() > 0, f"Loss should be positive, got {loss.item()} — check that you're computing MSE, not just subtraction"
    print(f"Loss ({mode}-prediction, zero model): {loss.item():.4f} ✓")

print("All prediction modes working!")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def diffusion_loss(
    model: nn.Module,
    x_0: torch.Tensor,
    sched: dict,
    T: int = 1000,
    prediction_mode: str = "eps",
) -> torch.Tensor:
    """Compute diffusion training loss with configurable prediction mode.
    
    Args:
        model: network that takes (x_t, t) and returns prediction
        x_0: clean data, shape (B, C, H, W)
        sched: schedule dict
        T: number of timesteps
        prediction_mode: one of "eps", "x0", "v"
    Returns:
        loss: scalar MSE
    """
    batch_size = x_0.shape[0]
    t = torch.randint(0, T, (batch_size,), device=x_0.device)               # (B,)
    eps = torch.randn_like(x_0)                                               # (B, C, H, W)
    
    sqrt_ab = sched['sqrt_alpha_bar'][t].view(-1, 1, 1, 1)                   # (B, 1, 1, 1)
    sqrt_1mab = sched['sqrt_one_minus_alpha_bar'][t].view(-1, 1, 1, 1)       # (B, 1, 1, 1)
    x_t = sqrt_ab * x_0 + sqrt_1mab * eps                                    # (B, C, H, W)
    
    prediction = model(x_t, t)                                                # (B, C, H, W)
    
    if prediction_mode == "eps":
        target = eps                                                           # (B, C, H, W)
    elif prediction_mode == "x0":
        target = x_0                                                           # (B, C, H, W)
    elif prediction_mode == "v":
        target = sqrt_ab * eps - sqrt_1mab * x_0                              # (B, C, H, W)
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")
    
    return F.mse_loss(prediction, target)                                      # scalar


# Test all three modes
torch.manual_seed(42)
for mode in ["eps", "x0", "v"]:
    loss = diffusion_loss(dummy_model, x_0_batch, sched, T=1000, prediction_mode=mode)
    print(f"Loss ({mode}-prediction, zero model): {loss.item():.4f}")

## 5.8 — The SNR Perspective

Let's look at diffusion through a single, unifying lens: the **Signal-to-Noise Ratio** ([Kingma & Gao 2023](https://arxiv.org/abs/2303.00848)). This is the most useful number for reasoning about what the model "sees" at each timestep.

At timestep $t$, the noisy sample is:

$$x_t = \underbrace{\sqrt{\bar{\alpha}_t}}_{\text{signal coeff}} x_0 + \underbrace{\sqrt{1 - \bar{\alpha}_t}}_{\text{noise coeff}} \varepsilon$$

The SNR is the ratio of signal power to noise power:

$$\text{SNR}(t) = \frac{\bar{\alpha}_t}{1 - \bar{\alpha}_t}$$

### What your image looks like at different SNR values

- **High SNR** (early timesteps, e.g. $t = 10$): mostly signal, little noise. You can still clearly see the digit. The pixel values are barely disturbed.
- **SNR = 1** (the crossover point): equal parts signal and noise. The image is hazy — you might guess it's a "7" but you're not sure.
- **Low SNR** (late timesteps, e.g. $t = 900$): mostly noise, almost no signal left. The image looks like TV static.

### Why SNR matters for training

The ELBO loss weights each timestep by $\frac{d}{dt}\text{SNR}(t)$, which is why different schedules lead to different implicit weightings. The simplified loss (uniform weighting) upweights high-SNR timesteps relative to the ELBO.

In [ ]:
# --- 5.8 SNR computation and visualization ---

def compute_snr(alpha_bar: torch.Tensor) -> torch.Tensor:
    """Compute signal-to-noise ratio: SNR(t) = alpha_bar / (1 - alpha_bar)."""
    return alpha_bar / (1.0 - alpha_bar)  # (T,)


# Compute SNR for linear schedule
snr_linear = compute_snr(sched['alpha_bar'])  # (1000,)

# Also compute cosine schedule for comparison (we'll define it fully in 5.9)
def cosine_alpha_bar(T: int, s: float = 0.008) -> torch.Tensor:
    """Compute alpha_bar using the cosine schedule from Nichol & Dhariwal 2021."""
    steps = torch.arange(T + 1, dtype=torch.float64)
    f_t = torch.cos(((steps / T) + s) / (1 + s) * (math.pi / 2)) ** 2
    alpha_bar = f_t[1:] / f_t[0]  # (T,)
    return alpha_bar.float()

alpha_bar_cosine = cosine_alpha_bar(1000)
snr_cosine = compute_snr(alpha_bar_cosine)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].plot(timesteps.numpy(), snr_linear.numpy(), label='Linear schedule', color='tab:blue')
axes[0].plot(timesteps.numpy(), snr_cosine.numpy(), label='Cosine schedule', color='tab:orange')
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='SNR = 1')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('SNR(t)')
axes[0].set_title('SNR (linear scale)')
axes[0].legend()
axes[0].set_ylim(0, 50)

# Log scale -- much more informative
axes[1].semilogy(timesteps.numpy(), snr_linear.numpy(), label='Linear schedule', color='tab:blue')
axes[1].semilogy(timesteps.numpy(), snr_cosine.numpy(), label='Cosine schedule', color='tab:orange')
axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='SNR = 1')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('SNR(t) [log scale]')
axes[1].set_title('SNR (log scale)')
axes[1].legend()

plt.tight_layout()
plt.show()

# What does the model "see" at different SNR levels?
x_0 = make_checkerboard()  # (1, 32, 32)
snr_targets = [100, 10, 1, 0.1, 0.01]

fig, axes = plt.subplots(1, len(snr_targets), figsize=(14, 3))
torch.manual_seed(42)

for ax, target_snr in zip(axes, snr_targets):
    # alpha_bar = snr / (1 + snr)
    ab = target_snr / (1.0 + target_snr)
    eps = torch.randn_like(x_0)
    x_noisy = math.sqrt(ab) * x_0 + math.sqrt(1 - ab) * eps
    ax.imshow(x_noisy[0].numpy(), cmap='gray', vmin=-3, vmax=3)
    ax.set_title(f"SNR = {target_snr}")
    ax.axis('off')

fig.suptitle("What the model sees at different SNR levels", fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 5.4: SNR Computation and SNR-Weighted Loss

**Part 1:** Find the timestep where SNR is closest to 1 for the linear schedule. This is the crossover point where signal and noise are balanced — the image transitions from "recognizable but noisy" to "mostly static."

**Part 2:** Implement an SNR-weighted loss where each timestep's loss is weighted by $\text{SNR}(t) / (1 + \text{SNR}(t))$.

This approximates the ELBO weighting and gives more importance to low-SNR (high-noise) timesteps, where the network has to work hardest.

**Hint:** $\text{SNR}(t) / (1 + \text{SNR}(t))$ simplifies to just $\bar{\alpha}_t$ — try working this out on paper.

In [ ]:
# YOUR CODE HERE — Exercise 5.4

# Part 1: Find the timestep where SNR = 1
snr_values = compute_snr(sched['alpha_bar'])  # (1000,)

# ===================== YOUR CODE HERE =====================
snr_eq_1_idx = None  # Find the index where SNR is closest to 1
# ====================== END YOUR CODE ======================

assert snr_eq_1_idx is not None, "Set snr_eq_1_idx — find where snr_values is closest to 1.0 (think argmin of the distance)"
print(f"Timestep closest to SNR=1: t={snr_eq_1_idx + 1}")
print(f"  SNR at that timestep: {snr_values[snr_eq_1_idx]:.4f}")
print(f"  alpha_bar: {sched['alpha_bar'][snr_eq_1_idx]:.4f} (should be ~0.5)")


# Part 2: SNR-weighted loss
def diffusion_loss_snr_weighted(
    model: nn.Module,
    x_0: torch.Tensor,
    sched: dict,
    T: int = 1000,
) -> torch.Tensor:
    """Compute SNR-weighted diffusion loss.
    
    Weight each sample's loss by SNR(t) / (1 + SNR(t)) = alpha_bar_t.
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests
torch.manual_seed(42)
loss_snr = diffusion_loss_snr_weighted(dummy_model, x_0_batch, sched, T=1000)
assert loss_snr is not None, "diffusion_loss_snr_weighted returned None — did you forget the return statement?"
assert loss_snr.item() > 0, f"Loss should be positive, got {loss_snr.item()} — are you computing per-sample MSE before weighting?"
loss_simple_test = diffusion_loss_simple(dummy_model, x_0_batch, sched, T=1000)
assert loss_snr.item() < loss_simple_test.item(), \
    f"SNR-weighted loss ({loss_snr.item():.4f}) should be smaller than simple loss ({loss_simple_test.item():.4f}) — alpha_bar weights downweight high-noise timesteps"
print(f"\nSimple loss:       {loss_simple_test.item():.4f}")
print(f"SNR-weighted loss: {loss_snr.item():.4f} ✓")
print("SNR-weighted loss is smaller because it downweights high-noise timesteps.")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# Part 1: Find timestep where SNR = 1
snr_values = compute_snr(sched['alpha_bar'])                      # (1000,)
snr_eq_1_idx = (snr_values - 1.0).abs().argmin().item()           # scalar index
print(f"Timestep closest to SNR=1: t={snr_eq_1_idx + 1}")
print(f"  SNR at that timestep: {snr_values[snr_eq_1_idx]:.4f}")
print(f"  alpha_bar at that timestep: {sched['alpha_bar'][snr_eq_1_idx]:.4f}")
print(f"  (alpha_bar should be ~0.5 when SNR=1, since SNR = ab/(1-ab) = 1 => ab = 0.5)")


# Part 2: SNR-weighted loss
def diffusion_loss_snr_weighted(
    model: nn.Module,
    x_0: torch.Tensor,
    sched: dict,
    T: int = 1000,
) -> torch.Tensor:
    """Compute SNR-weighted diffusion loss.
    
    Weight = SNR(t) / (1 + SNR(t)) = alpha_bar_t  (after simplification!)
    """
    batch_size = x_0.shape[0]
    t = torch.randint(0, T, (batch_size,), device=x_0.device)               # (B,)
    eps = torch.randn_like(x_0)                                               # (B, C, H, W)
    
    sqrt_ab = sched['sqrt_alpha_bar'][t].view(-1, 1, 1, 1)                   # (B, 1, 1, 1)
    sqrt_1mab = sched['sqrt_one_minus_alpha_bar'][t].view(-1, 1, 1, 1)       # (B, 1, 1, 1)
    x_t = sqrt_ab * x_0 + sqrt_1mab * eps                                    # (B, C, H, W)
    
    eps_theta = model(x_t, t)                                                 # (B, C, H, W)
    
    # Per-sample MSE
    per_sample_loss = (eps - eps_theta).pow(2).mean(dim=(1, 2, 3))           # (B,)
    
    # Weight by SNR(t) / (1 + SNR(t)) = alpha_bar_t
    weights = sched['alpha_bar'][t]                                           # (B,)
    
    weighted_loss = (weights * per_sample_loss).mean()                        # scalar
    return weighted_loss


torch.manual_seed(42)
loss_simple = diffusion_loss_simple(dummy_model, x_0_batch, sched, T=1000)
loss_snr = diffusion_loss_snr_weighted(dummy_model, x_0_batch, sched, T=1000)
print(f"\nSimple loss:       {loss_simple.item():.4f}")
print(f"SNR-weighted loss: {loss_snr.item():.4f}")
print("SNR-weighted loss is smaller because it downweights high-noise timesteps.")

## 5.9 — Variance Schedules

The choice of $\beta_t$ schedule has a big impact on training and sample quality. Let's look at the three most common options and understand what each one does to your image.

### Linear Schedule (DDPM)

$$\beta_t = \beta_{\text{start}} + \frac{t-1}{T-1}(\beta_{\text{end}} - \beta_{\text{start}})$$

Simple and effective, but it destroys information too quickly at high $t$. The transition from "mostly signal" to "mostly noise" is abrupt — the image goes from recognizable to static over a narrow range of timesteps, wasting capacity at both ends.

### Cosine Schedule ([Nichol & Dhariwal 2021](https://arxiv.org/abs/2102.09672))

$$\bar{\alpha}_t = \frac{f(t)}{f(0)}, \qquad f(t) = \cos\!\left(\frac{t/T + s}{1+s} \cdot \frac{\pi}{2}\right)^2$$

with small offset $s = 0.008$ to prevent $\beta_t$ from being too small near $t=0$.

This is designed so that $\bar{\alpha}_t$ follows a cosine curve, giving a **smoother** transition. The pixel values get erased more gradually — the image spends more timesteps in the "partially noisy" regime where the network can actually learn useful structure.

### Sigmoid Schedule

$$\beta_t = \sigma(-a + 2a \cdot t/T) \cdot (\beta_{\text{end}} - \beta_{\text{start}}) + \beta_{\text{start}}$$

where $\sigma$ is the sigmoid function and $a$ controls the steepness. This gives a smooth S-curve that offers a middle ground between linear and cosine.

In [ ]:
# --- 5.9 Implement and compare variance schedules ---

def linear_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> torch.Tensor:
    """Linear schedule: betas linearly interpolated from beta_start to beta_end."""
    return torch.linspace(beta_start, beta_end, T)  # (T,)


def cosine_beta_schedule(T: int, s: float = 0.008) -> torch.Tensor:
    """Cosine schedule from Nichol & Dhariwal 2021.
    
    Defines alpha_bar via cosine, then derives betas.
    """
    steps = torch.arange(T + 1, dtype=torch.float64)
    f_t = torch.cos(((steps / T) + s) / (1 + s) * (math.pi / 2)) ** 2      # (T+1,)
    alpha_bar = (f_t[1:] / f_t[0])                                           # (T,)
    # Derive betas: beta_t = 1 - alpha_bar_t / alpha_bar_{t-1}
    alpha_bar_full = torch.cat([torch.tensor([1.0], dtype=torch.float64), alpha_bar])  # (T+1,)
    betas = 1.0 - (alpha_bar_full[1:] / alpha_bar_full[:-1])                 # (T,)
    betas = torch.clamp(betas, min=0.0, max=0.999)                           # clip for stability
    return betas.float()                                                      # (T,)


def sigmoid_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02, steepness: float = 6.0) -> torch.Tensor:
    """Sigmoid schedule: smooth S-curve interpolation."""
    t = torch.linspace(-steepness, steepness, T)                              # (T,)
    betas = torch.sigmoid(t) * (beta_end - beta_start) + beta_start          # (T,)
    return betas                                                              # (T,)


T = 1000
betas_linear = linear_beta_schedule(T)
betas_cosine = cosine_beta_schedule(T)
betas_sigmoid = sigmoid_beta_schedule(T)

sched_linear = compute_schedule(betas_linear)
sched_cosine = compute_schedule(betas_cosine)
sched_sigmoid = compute_schedule(betas_sigmoid)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot betas
for name, s, color in [('Linear', sched_linear, 'tab:blue'), 
                         ('Cosine', sched_cosine, 'tab:orange'),
                         ('Sigmoid', sched_sigmoid, 'tab:green')]:
    axes[0].plot(timesteps.numpy(), s['betas'].numpy(), label=name, color=color)
    axes[1].plot(timesteps.numpy(), s['alpha_bar'].numpy(), label=name, color=color)
    axes[2].semilogy(timesteps.numpy(), compute_snr(s['alpha_bar']).numpy(), label=name, color=color)

axes[0].set_title(r'$\beta_t$')
axes[0].set_xlabel('Timestep')
axes[0].legend()

axes[1].set_title(r'$\bar{\alpha}_t$')
axes[1].set_xlabel('Timestep')
axes[1].legend()

axes[2].set_title('SNR (log scale)')
axes[2].set_xlabel('Timestep')
axes[2].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- 5.9 Visual comparison: how each schedule noises an image ---

x_0 = make_checkerboard()  # (1, 32, 32)
torch.manual_seed(42)

vis_steps = [0, 200, 400, 600, 800, 1000]
schedules = {'Linear': sched_linear, 'Cosine': sched_cosine, 'Sigmoid': sched_sigmoid}

fig, axes = plt.subplots(3, len(vis_steps), figsize=(16, 8))

for row, (name, s) in enumerate(schedules.items()):
    for col, t in enumerate(vis_steps):
        if t == 0:
            img = x_0[0]
        else:
            torch.manual_seed(42)  # Same noise for fair comparison
            eps = torch.randn_like(x_0)
            t_idx = t - 1
            img = (s['sqrt_alpha_bar'][t_idx] * x_0 + s['sqrt_one_minus_alpha_bar'][t_idx] * eps)[0]
        
        axes[row, col].imshow(img.numpy(), cmap='gray', vmin=-3, vmax=3)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f"t = {t}", fontsize=11)
    axes[row, 0].set_ylabel(name, fontsize=12, rotation=0, labelpad=50)

fig.suptitle("Forward Noising Under Different Schedules (same noise realization)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Exercise 5.5: Implement All Three Schedules and Compare

**Implement `linear_beta_schedule`, `cosine_beta_schedule`, and `sigmoid_beta_schedule` from scratch.** Then create two comparison plots.

### Plot 1: $\bar{\alpha}_t$ for all three schedules

This shows how much original signal survives at each timestep. You'll see that cosine keeps signal around much longer.

### Plot 2: $\log(\text{SNR}(t))$ for all three schedules

This makes differences visually clear. Look for where each curve crosses zero — that's where signal and noise are equal.

**Hints:**
- For cosine: define $\bar{\alpha}_t$ via the cosine formula, then derive $\beta_t = 1 - \bar{\alpha}_t / \bar{\alpha}_{t-1}$. Clamp betas to $[0, 0.999]$ for stability.
- For sigmoid: use `torch.sigmoid` on a linearly spaced range $[-a, a]$ to create the S-curve.

In [ ]:
# YOUR CODE HERE — Exercise 5.5

def linear_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> torch.Tensor:
    """Linear schedule: betas linearly interpolated from beta_start to beta_end."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


def cosine_beta_schedule(T: int, s: float = 0.008) -> torch.Tensor:
    """Cosine schedule from Nichol & Dhariwal 2021. Defines alpha_bar via cosine, derives betas."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


def sigmoid_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02, steepness: float = 6.0) -> torch.Tensor:
    """Sigmoid schedule: smooth S-curve interpolation."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
T_test = 1000
for name, fn in [("linear", linear_beta_schedule), ("cosine", cosine_beta_schedule), ("sigmoid", sigmoid_beta_schedule)]:
    betas = fn(T_test)
    assert betas is not None, f"{name}_beta_schedule returned None — did you forget the return statement?"
    assert betas.shape == (T_test,), \
        f"{name} shape should be ({T_test},), got {betas.shape} — make sure you return a 1-D tensor"
    assert (betas > 0).all(), \
        f"{name} has non-positive betas — all beta values must be > 0"
    assert (betas < 1).all(), \
        f"{name} has betas >= 1 — did you forget to clamp? betas should be small values like 0.0001 to 0.02"
    s = compute_schedule(betas)
    assert s['alpha_bar'][0] > 0.99, \
        f"{name} alpha_bar[0] should be ~1.0, got {s['alpha_bar'][0]:.4f} — the first step should barely change the image"
    assert s['alpha_bar'][-1] < 0.05, \
        f"{name} alpha_bar[-1] should be ~0, got {s['alpha_bar'][-1]:.4f} — by the last step the image should be nearly pure noise"
    print(f"{name:>8s}: alpha_bar[0]={s['alpha_bar'][0]:.4f}, alpha_bar[-1]={s['alpha_bar'][-1]:.6f} ✓")

print("\nAll schedules implemented correctly!")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
# (The schedule functions were already implemented above. Here we create the comparison plots.)

T = 1000
schedules_compare = {
    'Linear': compute_schedule(linear_beta_schedule(T)),
    'Cosine': compute_schedule(cosine_beta_schedule(T)),
    'Sigmoid': compute_schedule(sigmoid_beta_schedule(T)),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
t_axis = torch.arange(1, T + 1).numpy()

for name, s in schedules_compare.items():
    ab = s['alpha_bar']
    snr = compute_snr(ab)
    log_snr = torch.log(snr + 1e-10)
    
    axes[0].plot(t_axis, ab.numpy(), label=name)
    axes[1].plot(t_axis, log_snr.numpy(), label=name)

axes[0].set_title(r'$\bar{\alpha}_t$ Comparison')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel(r'$\bar{\alpha}_t$')
axes[0].legend()

axes[1].set_title(r'$\log\,\mathrm{SNR}(t)$ Comparison')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel(r'$\log\,\mathrm{SNR}$')
axes[1].axhline(y=0.0, color='gray', linestyle='--', alpha=0.5, label='SNR=1')
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary statistics
print(f"{'Schedule':<10} {'SNR=1 at t':<12} {'alpha_bar[T]':<14} {'alpha_bar[1]':<14}")
print("-" * 50)
for name, s in schedules_compare.items():
    snr = compute_snr(s['alpha_bar'])
    t_snr1 = (snr - 1.0).abs().argmin().item() + 1
    print(f"{name:<10} {t_snr1:<12} {s['alpha_bar'][-1].item():<14.6f} {s['alpha_bar'][0].item():<14.6f}")

### You've built the complete DDPM math toolkit

Let's take stock of what you now have:

| Component | What you implemented | Where it's used |
|---|---|---|
| `compute_schedule` | All schedule quantities from $\beta_t$ | Every training loop and sampler |
| Three prediction modes | eps, x0, v conversions | Flexible loss functions |
| `diffusion_loss` | Training loss with configurable parameterization | Training loop (Module 6) |
| SNR-weighted loss | ELBO-approximate weighting | Advanced training strategies |
| Three variance schedules | Linear, cosine, sigmoid | Schedule selection |

You've derived the ELBO, understood why "predict the noise" works, and implemented the loss function that powers real diffusion models. In the next module, we'll put all of this to work and train a model that actually generates images.

In [ ]:
# --- Celebration: the complete forward-reverse cycle visualized ---
# Let's put everything together — noise an image with each schedule,
# then show what the model would need to "see" at the SNR=1 crossover point.

x_0 = make_checkerboard()  # (1, 32, 32)
schedules_demo = {
    'Linear': compute_schedule(linear_beta_schedule(1000)),
    'Cosine': compute_schedule(cosine_beta_schedule(1000)),
    'Sigmoid': compute_schedule(sigmoid_beta_schedule(1000)),
}

fig, axes = plt.subplots(3, 5, figsize=(14, 8))
snr_levels = [10.0, 3.0, 1.0, 0.3, 0.1]

for row, (name, s) in enumerate(schedules_demo.items()):
    snr_vals = compute_snr(s['alpha_bar'])
    for col, target_snr in enumerate(snr_levels):
        t_idx = (snr_vals - target_snr).abs().argmin().item()
        torch.manual_seed(42)
        eps = torch.randn_like(x_0)
        x_t = s['sqrt_alpha_bar'][t_idx] * x_0 + s['sqrt_one_minus_alpha_bar'][t_idx] * eps
        
        axes[row, col].imshow(x_t[0].numpy(), cmap='gray', vmin=-3, vmax=3)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f"SNR = {target_snr}", fontsize=11)
    axes[row, 0].set_ylabel(name, fontsize=12, rotation=0, labelpad=55)

fig.suptitle("What the Model Sees: Same Image at Different SNR Levels Across Schedules", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("If the images above show clean patterns on the left gradually dissolving")
print("into noise on the right, your schedule toolkit is working perfectly.")

---

## Math Reinforcement Exercises

Now let's make sure the formulas really feel concrete. In these exercises, you'll numerically verify the key results we derived — running the sequential chain to confirm the closed-form shortcut, checking the posterior, and comparing schedules head-to-head.

### Exercise R1: Verify the Closed-Form $x_t$ Formula

**Verify numerically that $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon$ produces the same distribution as running the sequential chain.**

Here's the approach:
1. Fix a pixel value $x_0$ and a target timestep $t$
2. Run the sequential chain many times, collecting the distribution of values at step $t$
3. Compare the empirical mean and variance to the closed-form predictions: mean $= \sqrt{\bar{\alpha}_t}\, x_0$, variance $= 1 - \bar{\alpha}_t$

This should be your first easy win — if you got Exercise 5.1 right, this is just plugging in numbers.

In [ ]:
# YOUR CODE HERE — Exercise R1

betas_r1 = linear_beta_schedule(T=1000)
sched_r1 = compute_schedule(betas_r1)

x0_val = 0.8  # A fixed pixel value
target_t = 300
n_trials = 10000

# ===================== YOUR CODE HERE =====================
# 1. Run the sequential chain n_trials times for a single pixel.
#    At each trial, start from x0_val and apply forward_step for target_t steps.
#    Collect the final values.
sequential_samples = None  # Should be a tensor of shape (n_trials,)

# 2. Generate closed-form samples (much simpler — just use the formula directly)
closed_form_samples = None  # Should be a tensor of shape (n_trials,)
# ====================== END YOUR CODE ======================

# Verification
ab_t = sched_r1['alpha_bar'][target_t - 1].item()
expected_mean = math.sqrt(ab_t) * x0_val
expected_var = 1 - ab_t

assert sequential_samples is not None, "Compute sequential_samples — loop over n_trials, applying forward_step target_t times each"
assert closed_form_samples is not None, "Compute closed_form_samples — use sqrt(ab)*x0 + sqrt(1-ab)*randn(n_trials)"
assert abs(sequential_samples.mean().item() - expected_mean) < 0.05, \
    f"Sequential mean {sequential_samples.mean():.4f} too far from expected {expected_mean:.4f} — are you using the right betas at each step?"
assert abs(closed_form_samples.mean().item() - expected_mean) < 0.05, \
    f"Closed-form mean {closed_form_samples.mean():.4f} too far from expected {expected_mean:.4f} — check your sqrt(alpha_bar) coefficient"

print(f"Theoretical mean:  {expected_mean:.4f}")
print(f"Theoretical var:   {expected_var:.4f}")
print(f"Sequential  — mean: {sequential_samples.mean():.4f}, var: {sequential_samples.var():.4f}")
print(f"Closed-form — mean: {closed_form_samples.mean():.4f}, var: {closed_form_samples.var():.4f}")
print("Distributions match ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

betas_r1 = linear_beta_schedule(T=1000)
sched_r1 = compute_schedule(betas_r1)

x0_val = 0.8  # A fixed pixel value
target_t = 300
n_trials = 10000

# Method A: Sequential chain (many trials)
sequential_samples = []
for _ in range(n_trials):
    x = x0_val
    for step in range(target_t):
        beta = betas_r1[step].item()
        x = math.sqrt(1 - beta) * x + math.sqrt(beta) * torch.randn(1).item()
    sequential_samples.append(x)
sequential_samples = torch.tensor(sequential_samples)  # (n_trials,)

# Method B: Closed-form (many trials)
ab_t = sched_r1['alpha_bar'][target_t - 1].item()
closed_form_samples = math.sqrt(ab_t) * x0_val + math.sqrt(1 - ab_t) * torch.randn(n_trials)  # (n_trials,)

# Compare
print(f"Target: t={target_t}, x_0={x0_val}")
print(f"Theoretical mean:  {math.sqrt(ab_t) * x0_val:.4f}")
print(f"Theoretical var:   {1 - ab_t:.4f}")
print()
print(f"Sequential  -- mean: {sequential_samples.mean():.4f}, var: {sequential_samples.var():.4f}")
print(f"Closed-form -- mean: {closed_form_samples.mean():.4f}, var: {closed_form_samples.var():.4f}")

# Histogram comparison
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sequential_samples.numpy(), bins=60, alpha=0.5, density=True, label='Sequential chain')
ax.hist(closed_form_samples.numpy(), bins=60, alpha=0.5, density=True, label='Closed-form')
ax.set_title(f'Distribution of $x_{{t={target_t}}}$ for $x_0={x0_val}$')
ax.set_xlabel('$x_t$')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise R2: Verify the Posterior $q(x_{t-1} | x_t, x_0)$

**Verify that the closed-form posterior is correct by sampling from it and checking the statistics match.**

Here's the plan:
1. Fix $x_0$ and compute $x_t$ using the closed-form forward process
2. Compute the posterior mean $\tilde{\mu}_t$ and variance $\tilde{\beta}_t$ using the formulas from Section 5.5
3. Sample many $x_{t-1}$ values from $\mathcal{N}(\tilde{\mu}_t, \tilde{\beta}_t)$
4. Check that the empirical mean and variance match the theoretical values
5. Plot a histogram of the posterior samples overlaid with the theoretical Gaussian

In [ ]:
# YOUR CODE HERE — Exercise R2

betas_r2 = linear_beta_schedule(T=1000)
sched_r2 = compute_schedule(betas_r2)

t_idx = 50  # 0-indexed, so this is t=51
x0_val = torch.tensor([0.7])  # (1,) — scalar "image"

# Step 1: Compute x_t using closed-form
torch.manual_seed(99)
eps = torch.randn(1)
x_t_val = sched_r2['sqrt_alpha_bar'][t_idx] * x0_val + sched_r2['sqrt_one_minus_alpha_bar'][t_idx] * eps

# ===================== YOUR CODE HERE =====================
# Step 2: Compute the posterior mean and variance
#   Use the formula: mu = (sqrt(ab_prev)*beta_t)/(1-ab_t)*x0 + (sqrt(alpha_t)*(1-ab_prev))/(1-ab_t)*x_t
posterior_mean = None  # Use the formula from Section 5.5
posterior_var = None   # Use sched_r2['posterior_variance'][t_idx]

# Step 3: Sample from the posterior
n_samples = 50000
posterior_samples = None  # Shape: (n_samples,) — sample from N(posterior_mean, posterior_var)
# ====================== END YOUR CODE ======================

assert posterior_mean is not None, "Compute posterior_mean using the formula from Section 5.5"
assert posterior_var is not None, "Set posterior_var from sched_r2['posterior_variance'][t_idx]"
assert posterior_samples is not None, "Sample from N(posterior_mean, posterior_var) — use mean + sqrt(var) * randn(n_samples)"

print(f"At t_idx={t_idx}:")
print(f"  x_0 = {x0_val.item():.4f}")
print(f"  x_t = {x_t_val.item():.4f}")
print(f"  Posterior mean = {posterior_mean.item():.4f}")
print(f"  Posterior var  = {posterior_var:.6f}")
print(f"  Empirical mean: {posterior_samples.mean().item():.4f}")
print(f"  Empirical var:  {posterior_samples.var().item():.6f}")
assert abs(posterior_samples.mean().item() - posterior_mean.item()) < 0.01, \
    f"Empirical mean {posterior_samples.mean():.4f} doesn't match theoretical {posterior_mean.item():.4f} — check your mean formula coefficients"
assert abs(posterior_samples.var().item() - posterior_var) < 0.001, \
    f"Empirical variance {posterior_samples.var():.6f} doesn't match theoretical {posterior_var:.6f} — are you using sqrt(var) when sampling?"
print("Posterior verified ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# We verify the posterior analytically by direct computation, since
# rejection sampling is too expensive for high-dimensional data.
# Instead, we verify that sampling from the posterior and then applying
# one forward step recovers the correct joint distribution.

betas_r2 = linear_beta_schedule(T=1000)
sched_r2 = compute_schedule(betas_r2)

t_idx = 50  # Use a small t for clarity (0-indexed, so this is t=51)
x0_val = torch.tensor([0.7])       # (1,) -- scalar "image"

# Compute x_t using closed-form
torch.manual_seed(99)
eps = torch.randn(1)                # (1,)
x_t_val = sched_r2['sqrt_alpha_bar'][t_idx] * x0_val + sched_r2['sqrt_one_minus_alpha_bar'][t_idx] * eps  # (1,)

# Posterior mean (closed-form)
alpha_bar_t = sched_r2['alpha_bar'][t_idx].item()
alpha_bar_prev = sched_r2['alpha_bar_prev'][t_idx].item()
beta_t = sched_r2['betas'][t_idx].item()
alpha_t = sched_r2['alphas'][t_idx].item()

posterior_mean = (
    (math.sqrt(alpha_bar_prev) * beta_t) / (1 - alpha_bar_t) * x0_val
    + (math.sqrt(alpha_t) * (1 - alpha_bar_prev)) / (1 - alpha_bar_t) * x_t_val
)
posterior_var = sched_r2['posterior_variance'][t_idx].item()

print(f"At t_idx={t_idx}:")
print(f"  x_0 = {x0_val.item():.4f}")
print(f"  x_t = {x_t_val.item():.4f}")
print(f"  Posterior mean = {posterior_mean.item():.4f}")
print(f"  Posterior var  = {posterior_var:.6f}")
print(f"  Posterior std  = {math.sqrt(posterior_var):.6f}")

# Sample many x_{t-1} from the posterior and verify statistics
n_samples = 50000
posterior_samples = posterior_mean + math.sqrt(posterior_var) * torch.randn(n_samples)  # (n_samples,)

# Now verify: if we take these x_{t-1} and apply one forward step,
# we should get a distribution centered near x_t
x_t_reconstructed = math.sqrt(1 - beta_t) * posterior_samples + math.sqrt(beta_t) * torch.randn(n_samples)

print(f"\nVerification: forward step from posterior samples should match x_t")
print(f"  x_t (original):     {x_t_val.item():.4f}")
print(f"  E[forward(x_{{t-1}})]:{x_t_reconstructed.mean().item():.4f}")

# Also verify posterior sample statistics match
print(f"\nPosterior sample statistics:")
print(f"  Empirical mean: {posterior_samples.mean().item():.4f}  (expected {posterior_mean.item():.4f})")
print(f"  Empirical var:  {posterior_samples.var().item():.6f}  (expected {posterior_var:.6f})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(posterior_samples.numpy(), bins=80, density=True, alpha=0.6, label='Posterior samples')
# Overlay the theoretical Gaussian
x_range = torch.linspace(posterior_mean.item() - 4*math.sqrt(posterior_var),
                          posterior_mean.item() + 4*math.sqrt(posterior_var), 200)
pdf = torch.exp(-0.5 * ((x_range - posterior_mean.item()) / math.sqrt(posterior_var))**2) / (math.sqrt(2*math.pi*posterior_var))
ax.plot(x_range.numpy(), pdf.numpy(), 'r-', lw=2, label='Theoretical Gaussian')
ax.set_title(f'Posterior $q(x_{{t-1}} | x_t, x_0)$ at t={t_idx+1}')
ax.set_xlabel('$x_{t-1}$')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise R3: Find the Timestep Where SNR = 1

**For each of the three schedules (linear, cosine, sigmoid), find the exact timestep where SNR = 1.** At this crossover point, $\bar{\alpha}_t = 0.5$ — the image is half signal, half noise.

For each schedule:
1. Compute betas and the full schedule for $T = 1000$
2. Find the index where SNR is closest to 1
3. Print the timestep, $\bar{\alpha}_t$, and SNR at that point

Which schedule reaches the crossover latest? What does that tell you about how it distributes the noise budget?

In [ ]:
# YOUR CODE HERE — Exercise R3

T = 1000
schedule_fns = {
    'Linear': linear_beta_schedule,
    'Cosine': cosine_beta_schedule,
    'Sigmoid': sigmoid_beta_schedule,
}

# ===================== YOUR CODE HERE =====================
# For each schedule:
#   1. Compute betas and the full schedule
#   2. Compute SNR
#   3. Find the index where SNR is closest to 1
#   4. Print the results in a table
results = {}  # Store {name: (timestep, alpha_bar, snr)} for assertions

# ====================== END YOUR CODE ======================

# Tests — run this cell to check your work
assert len(results) == 3, f"Expected results for 3 schedules, got {len(results)} — compute results for all three"
for name in ['Linear', 'Cosine', 'Sigmoid']:
    assert name in results, f"Missing results for '{name}' schedule"
    t_val, ab_val, snr_val = results[name]
    assert 1 <= t_val <= T, f"{name}: timestep {t_val} out of range — should be between 1 and {T}"
    assert abs(ab_val - 0.5) < 0.05, f"{name}: alpha_bar={ab_val:.4f} at SNR=1 should be ~0.5 — check your SNR formula"
    assert abs(snr_val - 1.0) < 0.5, f"{name}: SNR={snr_val:.4f} should be close to 1.0"
    print(f"{name:>8s}: t={t_val}, alpha_bar={ab_val:.4f}, SNR={snr_val:.4f} ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

T = 1000
schedule_fns = {
    'Linear': linear_beta_schedule,
    'Cosine': cosine_beta_schedule,
    'Sigmoid': sigmoid_beta_schedule,
}

results = {}

print(f"{'Schedule':<10} {'t (SNR=1)':<12} {'alpha_bar':<12} {'SNR':<12}")
print("-" * 46)

for name, fn in schedule_fns.items():
    betas = fn(T)
    s = compute_schedule(betas)
    snr = compute_snr(s['alpha_bar'])
    
    # Find index closest to SNR = 1
    idx = (snr - 1.0).abs().argmin().item()
    results[name] = (idx + 1, s['alpha_bar'][idx].item(), snr[idx].item())
    
    print(f"{name:<10} {idx + 1:<12} {s['alpha_bar'][idx].item():<12.6f} {snr[idx].item():<12.6f}")

print("\nThe linear schedule reaches the SNR=1 crossover much earlier than cosine,")
print("meaning it wastes timesteps in the pure-noise regime where the model gets")
print("little learning signal. The cosine schedule distributes noise more evenly.")

---

### Module 5 complete

You've built the complete mathematical foundation for diffusion models. The key ideas to carry forward:

- The **closed-form forward process** lets us jump to any noise level in one step — no sequential chain needed for training
- The **ELBO derivation** reduces to a simple noise-prediction MSE loss
- The **SNR perspective** unifies everything: schedules, loss weighting, and what the model sees at each timestep
- **Cosine schedules** distribute noise more evenly than linear, giving the model more useful training signal

In Module 6, we'll take everything from this module and use it to train a real diffusion model on MNIST. The math you've verified here becomes working code.